# RAGTurk retrieval check

Puts our students next to the baselines on **RAGTurk** (METU NLP, SIGTURK 2026, [arXiv 2602.03652](https://arxiv.org/abs/2602.03652)): questions about Turkish Wikipedia articles, each linked to the 1-3 chunks that contain the answer. Retrieval only: the RAG pipeline half of the paper (HyDE, rerankers, generation) is not used.

**What is in the data we use** (`formal_5k` of `metunlp/ragturk`, revision `86d35335`, CC BY-NC-SA 4.0, kept private on Drive as `data/ragturk/ragturk_formal5k.json`):
- 2,790 Wikipedia articles (dump 2025-12-20), 24,103 chunks (median 594 characters), 5,579 questions (2 per article; FACTUAL 2,968, INTERPRETATION 2,611). The paper reports 4,891 / 42,304 / 9,777 for its Wikipedia part, so the release is about 57% of it; the web part (`informal_6k`) is not published.
- Questions were written by gpt-oss:120B and checked by gemini-2.5-flash; **no human verification** is reported. LLM-written questions tend to share words with their chunk, which favours lexical matching and compresses the gaps between models.
- Search runs over the whole corpus (all 24,103 chunks), not inside the question's own article.

**Statistics.** Two questions per article are not independent, so the intervals below resample *articles* (cluster bootstrap), not questions.

**Contamination.** The corpus is Wikipedia and so is part of our training mix; the last cells measure the 13-gram overlap per article and re-score without the exposed articles.

In [ ]:
import os, json, re, gc, time, shutil
from pathlib import Path
import numpy as np
import torch
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
os.environ['HF_HOME'] = '/content/hf_cache'      # local disk, never Drive
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    print('HF_TOKEN not set (only needed for gated models):', exc)

DATA_PATH = f'{DRIVE_ROOT}/data/ragturk/ragturk_formal5k.json'
DATA = json.load(open(DATA_PATH, encoding='utf-8'))
print(DATA['meta'])
corpus, queries = DATA['corpus'], DATA['queries']
chunk_index = {c['id']: i for i, c in enumerate(corpus)}
relevant = [np.array([chunk_index[r] for r in q['relevant']]) for q in queries]
categories = np.array([q['category'] for q in queries])
print(len(corpus), 'chunks,', len(queries), 'questions;', {c: int((categories == c).sum()) for c in sorted(set(categories))})

## Metrics and statistics

Relevance is binary: a chunk is relevant if the question lists it (1-3 chunks per question). `nDCG@10` is the headline; `Recall@5`, `nDCG@5`, `MRR` and `AP` follow the paper's list (their exact definitions are not in the paper, so numbers are comparable between our models, not to the paper's table).

In [ ]:
def metrics_from_topk(top, relevant):
    # top: (Q, K>=100) retrieved chunk indices in rank order; relevant: list of index arrays
    names = ('ndcg@10', 'ndcg@5', 'recall@5', 'recall@10', 'mrr@10', 'ap@100', 'hit@1')
    out = {m: np.zeros(len(relevant)) for m in names}
    disc = 1.0 / np.log2(np.arange(2, top.shape[1] + 2))
    for i, r in enumerate(relevant):
        hit = np.isin(top[i], r)
        R = len(r)
        for k, name in ((10, 'ndcg@10'), (5, 'ndcg@5')):
            out[name][i] = (hit[:k] * disc[:k]).sum() / disc[:min(R, k)].sum()
        out['recall@5'][i] = hit[:5].sum() / R
        out['recall@10'][i] = hit[:10].sum() / R
        first = np.flatnonzero(hit[:10])
        out['mrr@10'][i] = 1.0 / (first[0] + 1) if len(first) else 0.0
        out['hit@1'][i] = float(hit[0])
        pos = np.flatnonzero(hit)
        out['ap@100'][i] = (np.cumsum(hit)[pos] / (pos + 1)).sum() / R
    return out


def search(q_emb, d_emb, k=100, batch=512):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    d = torch.from_numpy(d_emb).to(device)
    tops = []
    for s in range(0, len(q_emb), batch):
        q = torch.from_numpy(q_emb[s:s + batch]).to(device)
        tops.append((q @ d.T).topk(k, dim=1).indices.cpu().numpy())
    return np.concatenate(tops)


_, article_group = np.unique(np.array([q['article_id'] for q in queries]), return_inverse=True)
N_ARTICLES = int(article_group.max()) + 1


def cluster_mean_ci(x, mask=None, n=5000, seed=0):
    # mean of x (optionally over a boolean subset) with a 95% interval from resampling ARTICLES
    w = np.ones(len(x)) if mask is None else mask.astype(float)
    sums = np.bincount(article_group, weights=x * w, minlength=N_ARTICLES)
    counts = np.bincount(article_group, weights=w, minlength=N_ARTICLES)
    idx = np.random.default_rng(seed).integers(0, N_ARTICLES, (n, N_ARTICLES))
    boot = sums[idx].sum(1) / np.maximum(counts[idx].sum(1), 1)
    return sums.sum() / counts.sum(), np.percentile(boot, [2.5, 97.5])

## Models

Same set as the turkish-rag-eval check plus the 8K-base student. `path` = a `final/` folder on Drive (loaded as transformer + mean pooling, the same protocol as TR-MTEB), `hf` = a Hugging Face repo at a pinned revision. `q` / `d` are query / document prefixes. Vectors are normalised here, so cosine = dot product for every model.

In [ ]:
INSTR_8K = 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:'      # no space after "Query:" (model card)
LAB_INSTR = 'Instruct: Given a Turkish search text, retrieve relevant passages written in Turkish that best match the search text.\nQuery: '
RUN = lambda name: f'{DRIVE_ROOT}/runs/{name}/final'

MODELS = {
    # --- our students ---
    'ours_pair_instr8k_s10800': dict(path=RUN('scale_two_instr8k_s10800_lr1e4/20261027-152516-b2b0c4b')),     # 1k base, BGE + instructed 8K, 10,800 steps (TR-MTEB 67.71)
    'ours_pair_plain_s10800':   dict(path=RUN('scale_two_s10800_lr1e4/20261027-152514-b2b0c4b')),            # (67.55)
    'ours_pair_plain_s5400':    dict(path=RUN('scale_two_lr1e4_mix200k_s5400/20261027-063854-79eb81a')),     # (67.16)
    'ours_single_bge_s5400':    dict(path=RUN('scale_bge_mix200k_s5400_lr1e4/20261027-101819-77232d7')),     # (66.07)
    'ours_single_8k_s5400':     dict(path=RUN('scale_8k_mix200k_s5400_lr1e4/20261027-105836-77232d7')),      # (65.67)
    'ours_base8k_pair_instr8k_s10800': dict(path=RUN('base8k_two_instr8k_s10800_lr1e4/20261029-111849-d50feeb')),   # same recipe, 8K base as the student
    # --- baselines ---
    'raw_base_1k':   dict(hf='ytu-ce-cosmos/modernbert-tr-base-1k', rev='c2e35f553101cce5b03fbb318ef8728473fd34dd'),
    'lab_1k_embed':  dict(hf='ytu-ce-cosmos/modernbert-tr-1k-embed', rev='ba1e17bc7e95a257b4e614c300c4ca076711fce6'),
    'lab_1k_embed_instr': dict(hf='ytu-ce-cosmos/modernbert-tr-1k-embed', rev='ba1e17bc7e95a257b4e614c300c4ca076711fce6', q=LAB_INSTR),
    'lab_8k_embed':  dict(hf='ytu-ce-cosmos/modernbert-tr-embed', rev='855820e0079102805908d60d32cd9fd59c970b80'),
    'lab_8k_embed_instr': dict(hf='ytu-ce-cosmos/modernbert-tr-embed', rev='855820e0079102805908d60d32cd9fd59c970b80', q=INSTR_8K),
    'bge_m3':        dict(hf='BAAI/bge-m3', rev='5617a9f61b028005a4858fdac845db406aefb181', fp16=True),
}
REFERENCE = 'ours_pair_instr8k_s10800'     # paired comparisons are against this model
TO_RUN = list(MODELS)                      # or a subset
DOC_FORMAT = 'content'                     # 'content' = the chunk text; 'title_path' = "title > heading path\n" + chunk text
for name, spec in MODELS.items():
    if 'path' in spec:
        assert os.path.isdir(spec['path']), f'{name}: {spec["path"]} not found'
print(len(MODELS), 'models')

## Run

One model after another in this process (the 150M-560M models fit side by side with room to spare; memory is released between models). Results go to Drive `eval_ragturk/<label>__<doc format>/`: `summary.json` and `perquery.npz`; a finished model is skipped on re-run.

In [ ]:
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

RESULTS = Path(f'{DRIVE_ROOT}/eval_ragturk')
RESULTS.mkdir(parents=True, exist_ok=True)


def doc_text(c):
    return f"{c['title']} > {c['heading_path']}\n{c['text']}" if DOC_FORMAT == 'title_path' else c['text']


def materialise(spec):
    if 'path' in spec:
        return spec['path']
    local = f"/content/models/{spec['hf'].replace('/', '__')}_{spec['rev'][:8]}"
    if not os.path.isdir(local):
        snapshot_download(spec['hf'], revision=spec['rev'], local_dir=local)
    return local


def run_model(label, spec):
    t0 = time.time()
    model = SentenceTransformer(materialise(spec), device='cuda')
    model.max_seq_length = 512
    if spec.get('fp16'):
        model.half()
    enc = lambda texts: model.encode(texts, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
    d_emb = enc([spec.get('d', '') + doc_text(c) for c in corpus])
    t_docs = time.time() - t0
    q_emb = enc([spec.get('q', '') + q['question'] for q in queries])
    del model
    gc.collect(); torch.cuda.empty_cache()
    top = search(q_emb, d_emb)
    m = metrics_from_topk(top, relevant)
    out = RESULTS / f'{label}__{DOC_FORMAT}'
    out.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out / 'perquery.npz', qids=np.array([q['qid'] for q in queries]), **m)
    summary = {'label': label, 'spec': {k: v for k, v in spec.items() if k != 'path'}, 'path': spec.get('path'), 'doc_format': DOC_FORMAT,
               'dim': int(d_emb.shape[1]), 'n_chunks': len(corpus), 'n_queries': len(queries),
               'seconds_total': round(time.time() - t0, 1), 'seconds_corpus': round(t_docs, 1),
               'mean': {k: float(v.mean()) for k, v in m.items()}, 'data_revision': DATA['meta']['revision']}
    json.dump(summary, open(out / 'summary.json', 'w', encoding='utf-8'), indent=1)
    return summary


failed = {}
for label in TO_RUN:
    if (RESULTS / f'{label}__{DOC_FORMAT}' / 'summary.json').exists():
        print(f'{label}: already done, skipped')
        continue
    print(f'########## {label}')
    try:
        s = run_model(label, MODELS[label])
        print(f"=== {label}: {s['seconds_total'] / 60:.1f} min   nDCG@10 {s['mean']['ndcg@10']:.3f}   Recall@5 {s['mean']['recall@5']:.3f}   MRR@10 {s['mean']['mrr@10']:.3f}")
    except Exception as exc:
        failed[label] = f'{type(exc).__name__}: {str(exc)[:300]}'
        print(f'=== {label} FAILED: {failed[label]}')
        gc.collect(); torch.cuda.empty_cache()
print('\nfailed:', failed or 'none')

## Results

Means over all 5,579 questions with 95% intervals from resampling articles; then the paired difference to the reference model (`*` = the interval excludes 0), overall and by question type.

In [ ]:
def load_results():
    res = {}
    for label in MODELS:
        p = RESULTS / f'{label}__{DOC_FORMAT}' / 'perquery.npz'
        if p.exists():
            z = np.load(p)
            assert list(z['qids']) == [q['qid'] for q in queries], f'{label}: different questions'
            res[label] = {k: z[k] for k in z.files if k != 'qids'}
    return res

res = load_results()
order = sorted(res, key=lambda l: -res[l]['ndcg@10'].mean())

print(f'{"model":<34}' + ''.join(f'{m:>10}' for m in ('nDCG@10', 'nDCG@5', 'Recall@5', 'MRR@10', 'AP@100', 'Hit@1')) + '   95% interval of nDCG@10')
for label in order:
    r = res[label]
    mean, (lo, hi) = cluster_mean_ci(r['ndcg@10'])
    print(f'{label:<34}' + ''.join(f'{r[m].mean():10.3f}' for m in ('ndcg@10', 'ndcg@5', 'recall@5', 'mrr@10', 'ap@100', 'hit@1')) + f'   [{lo:.3f}, {hi:.3f}]')

print(f'\npaired difference in nDCG@10: model minus {REFERENCE} (negative = the model is worse than the reference)')
print(f'{"model":<34}{"all":>9}{"95% interval":>20}{"FACTUAL":>10}{"INTERPR.":>10}')
ref = res[REFERENCE]['ndcg@10']
for label in order:
    if label == REFERENCE:
        continue
    d = res[label]['ndcg@10'] - ref
    mean, (lo, hi) = cluster_mean_ci(d)
    mark = '  *' if (lo > 0 or hi < 0) else ''
    by_cat = [cluster_mean_ci(d, categories == c) for c in ('FACTUAL', 'INTERPRETATION')]
    cells_ = ''.join(f'{m:+10.3f}' + ('*' if (ci[0] > 0 or ci[1] < 0) else ' ') for m, ci in by_cat)
    print(f'{label:<34}{mean:+9.3f}{f"[{lo:+.3f}, {hi:+.3f}]":>20}{cells_}{mark}')

## Training-data overlap

Per article: the share of its 13-grams (lower-cased word sequences) that occur anywhere in our 200K training mix. The mix contains Wikipedia articles and web text, so some articles may have been seen. An article counts as *exposed* above the threshold; the table then re-scores the models on the questions of the other (clean) articles. Exact 13-grams only: paraphrases and the teachers' own training data are not measured.

In [ ]:
N, THRESHOLD = 13, 0.05
tok = lambda text: re.findall(r'\w+', text.lower())

article_text = {}
for c in corpus:
    article_text.setdefault(c['article_id'], []).append(c['text'])
owner, total = {}, {}
for aid, parts in article_text.items():
    words = tok(' '.join(parts))
    grams = {hash(tuple(words[i:i + N])) for i in range(len(words) - N + 1)}
    total[aid] = len(grams)
    for g in grams:
        owner.setdefault(g, aid)
print(f'{len(article_text)} articles, {len(owner)} distinct {N}-grams')

hit_texts, n_texts, found = 0, 0, {}
with open(f'{DRIVE_ROOT}/data/mix/mix_v2_200k.jsonl', encoding='utf-8') as f:
    for line in f:
        n_texts += 1
        words = tok(json.loads(line)['text'])
        hits = {hash(tuple(words[i:i + N])) for i in range(len(words) - N + 1)} & owner.keys()
        if hits:
            hit_texts += 1
            for g in hits:
                found.setdefault(owner[g], set()).add(g)
covered = sum(len(v) for v in found.values())
print(f'{n_texts} mix texts; {hit_texts} share at least one {N}-gram with the RAGTurk articles')
print(f'{covered} of {len(owner)} article {N}-grams ({100 * covered / max(len(owner), 1):.2f}%) occur in the mix')
share = {aid: len(found.get(aid, ())) / max(t, 1) for aid, t in total.items()}
exposed_articles = {a for a, s in share.items() if s >= THRESHOLD}
exposed = np.array([q['article_id'] in exposed_articles for q in queries])
print(f'{len(exposed_articles)} of {len(share)} articles have >= {THRESHOLD:.0%} overlap; {int(exposed.sum())} of {len(queries)} questions point to them')
for s, a in sorted(((s, a) for a, s in share.items()), reverse=True)[:8]:
    print(f'  {s:6.1%}  {a}')

print(f'\n{"model":<34}{"all":>8}{"clean":>8}{"exposed":>9}   (nDCG@10; clean = without the exposed articles\' questions)')
for label in order:
    x = res[label]['ndcg@10']
    print(f'{label:<34}{x.mean():8.3f}{x[~exposed].mean():8.3f}{(x[exposed].mean() if exposed.any() else float("nan")):9.3f}')

print(f'\npaired difference to {REFERENCE} on the clean articles only')
ref_clean = res[REFERENCE]['ndcg@10']
for label in order:
    if label == REFERENCE:
        continue
    mean, (lo, hi) = cluster_mean_ci(res[label]['ndcg@10'] - ref_clean, ~exposed)
    print(f'{label:<34}{mean:+9.3f}   [{lo:+.3f}, {hi:+.3f}]{"  *" if (lo > 0 or hi < 0) else ""}')

In [ ]:
from google.colab import runtime
runtime.unassign()